# Phase 4: Causal Patching — Activation Patching

**Experimental protocol (Denoising):**
1. Run **clean prompt** → save clean hidden states for all layers (at the last token of `<hop1>`)
2. Run **counterfactual (corrupted) prompt** normally → save the baseline (hallucinated) answer
3. For **each layer**: Run **counterfactual prompt again**, but REPLACE the layer's hidden state (at the last token of `<hop1>`) with the CLEAN state harvested in step 1
4. Measure whether injecting the clean state successfully restores the correct answer

If the correct answer is restored in ≥50% of cases for a given layer, that layer is causally responsible for the hop-1 reasoning step.

In [ ]:
!pip install -q transformers accelerate huggingface_hub
import huggingface_hub
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("Logged in via Kaggle Secrets!")
except Exception as e:
    print(f"Kaggle Secrets failed ({e}), using fallback...")
    hf_token = "REPLACE_ME_WITH_YOUR_RAW_HF_TOKEN"
    huggingface_hub.login(token=hf_token)

In [ ]:
import torch
import numpy as np
import random

# Set seed for determinism
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

import json, torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print(f"Model loaded. Device: {model.device}")

In [ ]:
print("Downloading dataset...")
file_path = hf_hub_download(
    repo_id="AnishRacherla/LinguaFranca-Phase3",
    filename="data/2wikimultihopqa/augmented.jsonl",
    repo_type="dataset"
)
with open(file_path) as f:
    data = [json.loads(line) for line in f]

counterfactuals = [d for d in data if d.get("is_counterfactual", False)]
cleans = {d["id"]: d for d in data if not d.get("is_counterfactual", False)}
print(f"Loaded {len(cleans)} clean examples, {len(counterfactuals)} counterfactuals.")

# Build minimal pairs: clean example answered correctly (hop1 label=0) 
# AND its CF twin answered incorrectly (hop1 label=1)
pairs = []
for cf in counterfactuals:
    clean = cleans.get(cf.get("clean_pair_id"))
    if not clean:
        continue
    clean_h1 = next((h for h in clean.get("hops", []) if h["hop_idx"] == 1), None)
    cf_h1 = next((h for h in cf.get("hops", []) if h["hop_idx"] == 1), None)
    if clean_h1 and cf_h1 and clean_h1.get("label") == 0 and cf_h1.get("label") == 1:
        pairs.append((clean, cf))
            

print(f"Found {len(pairs)} perfect minimal pairs.")

In [ ]:
# ===== CELL 4 v3 — logit-based causal tracing =====
# v3: builds the corrupted prompt from the CLEAN prompt (only the question entity differs),
#     removing the spurious system-prompt date difference ('27' -> '30') seen in v2.
# Cells 1-3 unchanged (model, tokenizer, `pairs` must exist).

import re, json, difflib
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

num_layers = model.config.num_hidden_layers
DEV = model.device
WINDOW = 5          # MLP window: layers l-2 .. l+2
MIN_GAP = 1.0       # pair needs >= 1 nat clean-vs-corrupted gap to be a valid test
BATCH = 7           # patched runs per forward pass; lower to 4 if you hit CUDA OOM
RUN_MLP_WINDOW = True

def get_hop1(cot):
    m = re.search(r'(<hop1>.*?</hop1>)', cot, re.DOTALL)
    return m.group(1) if m else ""

def enc(text):
    return tokenizer(text, add_special_tokens=False, return_tensors="pt").input_ids[0]

def changed_blocks(a, b):
    """Token blocks where clean (a) and cf (b) prompts differ. Works even if lengths differ."""
    sm = difflib.SequenceMatcher(a=a.tolist(), b=b.tolist(), autojunk=False)
    return [(i1, i2, j1, j2) for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal"]

def make_store_hook(store, key):
    def hook(module, args, output):
        h = output[0] if isinstance(output, tuple) else output
        store[key] = h[0].detach().clone()
        return output
    return hook

@torch.no_grad()
def run_batch(input_ids, P, mask, configs):
    """configs: list (one per batch row) of [(module, cf_positions, clean_vectors), ...].
    Returns summed log P(target tokens under mask) for each row."""
    out = []
    tgt = input_ids[P:].to(DEV)
    for s in range(0, len(configs), BATCH):
        chunk = configs[s:s + BATCH]
        by_mod = {}
        for b, cfg in enumerate(chunk):
            for mod, pos, vecs in cfg:
                by_mod.setdefault(mod, []).append((b, pos, vecs))
        handles = []
        for mod, items in by_mod.items():
            def hook(module, args, output, items=items):
                h = output[0] if isinstance(output, tuple) else output
                h = h.clone()
                for b, pos, vecs in items:
                    h[b, pos, :] = vecs
                return (h,) + tuple(output[1:]) if isinstance(output, tuple) else h
            handles.append(mod.register_forward_hook(hook))
        try:
            ids = input_ids.unsqueeze(0).repeat(len(chunk), 1).to(DEV)
            hid = model.model(input_ids=ids, use_cache=False).last_hidden_state[:, P - 1:-1]
            lp = F.log_softmax(model.lm_head(hid).float(), dim=-1)
        finally:
            for hd in handles:
                hd.remove()
        lp = lp.gather(2, tgt.view(1, -1, 1).expand(len(chunk), -1, 1)).squeeze(2)
        out += lp[:, mask.to(DEV)].sum(1).tolist()
    return out

CONDITIONS = ["entity_single", "last_single"] + (["mlp_window"] if RUN_MLP_WINDOW else [])
results = []
skipped = {"no_hop1": 0, "no_diff": 0, "no_gap": 0}
n_len_mismatch_used = 0
n_prompt_fallback = 0

for idx, (clean, cf) in enumerate(tqdm(pairs, desc="Causal tracing")):
    tgt_text = get_hop1(clean["generated_cot"])          # the CORRECT hop 1 from the clean run
    if not tgt_text:
        skipped["no_hop1"] += 1; continue

    # Minimal pair: identical prompt except the question entity (no date / template drift)
    if clean["question"] in clean["prompt"]:
        cf_prompt = clean["prompt"].replace(clean["question"], cf["question"])
    else:
        cf_prompt = cf["prompt"]; n_prompt_fallback += 1
    c_ids, f_ids, t_ids = enc(clean["prompt"]), enc(cf_prompt), enc(tgt_text)
    blocks = changed_blocks(c_ids, f_ids)
    if not blocks:
        skipped["no_diff"] += 1; continue
    n_raw_blocks = len(blocks)
    # Merge into ONE swapped span (the question entity); a title like 'Call The X' vs 'Y The Z'
    # can otherwise be split around a shared token like ' The'.
    blocks = [(min(b[0] for b in blocks), max(b[1] for b in blocks),
               min(b[2] for b in blocks), max(b[3] for b in blocks))]
    if blocks[0][1] == blocks[0][0] or blocks[0][3] == blocks[0][2]:
        skipped["no_diff"] += 1; continue
    if len(c_ids) != len(f_ids):
        n_len_mismatch_used += 1

    if idx < 3:
        for i1, i2, j1, j2 in blocks:
            print(f"[pair {idx}] swapped: '{tokenizer.decode(c_ids[i1:i2])}' -> '{tokenizer.decode(f_ids[j1:j2])}'")

    # Patch site = LAST token of each swapped block (Meng et al.'s "last subject token")
    c_pos = torch.tensor([i2 - 1 for i1, i2, j1, j2 in blocks], device=DEV)
    f_pos = torch.tensor([j2 - 1 for i1, i2, j1, j2 in blocks], device=DEV)

    # Score only reasoning tokens of hop 1, not the copied entity-name tokens
    ent_tok = torch.cat([torch.cat([c_ids[i1:i2], f_ids[j1:j2]]) for i1, i2, j1, j2 in blocks])
    mask = ~torch.isin(t_ids, ent_tok)
    if mask.sum() == 0:
        skipped["no_gap"] += 1; continue

    Pc, Pf = len(c_ids), len(f_ids)
    clean_in, cf_in = torch.cat([c_ids, t_ids]), torch.cat([f_ids, t_ids])

    # 1) clean run: harvest residual stream + MLP outputs at every layer
    store = {}
    hs = []
    for l in range(num_layers):
        hs.append(model.model.layers[l].register_forward_hook(make_store_hook(store, ("res", l))))
        hs.append(model.model.layers[l].mlp.register_forward_hook(make_store_hook(store, ("mlp", l))))
    try:
        lp_clean = run_batch(clean_in, Pc, mask, [[]])[0]
    finally:
        for h in hs:
            h.remove()

    # 2) corrupted run, unpatched
    lp_cf = run_batch(cf_in, Pf, mask, [[]])[0]
    gap = lp_clean - lp_cf
    if gap < MIN_GAP:
        skipped["no_gap"] += 1; continue

    # 3) all patched runs for this pair
    last_c = torch.tensor([Pc - 1], device=DEV)
    last_f = torch.tensor([Pf - 1], device=DEV)
    configs, keys = [], []
    for l in range(num_layers):
        L = model.model.layers
        configs.append([(L[l], f_pos, store[("res", l)][c_pos])]); keys.append(("entity_single", l))
        configs.append([(L[l], last_f, store[("res", l)][last_c])]); keys.append(("last_single", l))
        if RUN_MLP_WINDOW:
            win = range(max(0, l - WINDOW // 2), min(num_layers, l + WINDOW // 2 + 1))
            configs.append([(L[k].mlp, f_pos, store[("mlp", k)][c_pos]) for k in win]); keys.append(("mlp_window", l))
    scores = run_batch(cf_in, Pf, mask, configs)

    rec = {k: [0.0] * num_layers for k in CONDITIONS}
    for (k, l), lp in zip(keys, scores):
        rec[k][l] = (lp - lp_cf) / gap              # 0 = no effect, 1 = fully restored
    rec["gap"] = gap
    rec["n_blocks"] = n_raw_blocks
    results.append(rec)
    del store
    torch.cuda.empty_cache()

# ===== Results =====
n = len(results)
print(f"\nPairs: {len(pairs)} | valid: {n} (of which {n_len_mismatch_used} had different prompt lengths) | skipped: {skipped}")
print(f"Prompt fallback (question not found verbatim in clean prompt): {n_prompt_fallback}")
print(f"Pairs whose diff had >1 raw block (merged into one span): {sum(r['n_blocks'] > 1 for r in results)}\n")
assert n > 0, "No valid pairs — check the skipped counts above."
R = {k: np.array([r[k] for r in results]) for k in CONDITIONS}

for k in CONDITIONS:
    mean = R[k].mean(0); ci = 1.96 * R[k].std(0) / np.sqrt(n); frac = (R[k] > 0.5).mean(0) * 100
    print(f"--- {k} ---")
    for l in range(num_layers):
        print(f"Layer {l:02d}: recovery {mean[l]:5.2f} ± {ci[l]:.2f} | pairs >50% restored: {frac[l]:5.1f}%")
    print()

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
for k in CONDITIONS:
    m = R[k].mean(0); c = 1.96 * R[k].std(0) / np.sqrt(n)
    plt.plot(range(num_layers), m, marker="o", label=k)
    plt.fill_between(range(num_layers), m - c, m + c, alpha=0.2)
plt.axhline(0, color="gray", lw=0.8)
plt.xlabel("Layer"); plt.ylabel("Fraction of clean-vs-corrupted gap recovered")
plt.title(f"Causal tracing of hop 1 (n={n} pairs)")
plt.legend(); plt.grid(alpha=0.3)
plt.savefig("causal_trace.png", dpi=200, bbox_inches="tight"); plt.show()

with open("causal_trace_results.json", "w") as f:
    json.dump({"skipped": skipped, "n": n, "n_len_mismatch_used": n_len_mismatch_used,
               "results": [{k: (v if k in ("gap", "n_blocks") else list(map(float, v))) for k, v in r.items()} for r in results]}, f)
print("Saved causal_trace_results.json and causal_trace.png")
